In [1]:
HYPERPARAMS = {
    "model_id": "microsoft/Phi-3-mini-4k-instruct",
    # LoRA: which part of the model we retrain
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    # Training
    "num_train_epochs": 10,
    "learning_rate": 3e-4,
    "per_device_train_batch_size": 16,
    "max_seq_length": 64,
    "validation_size": 0.1,
    "seed": 42,
}

In [2]:
from peft import LoraConfig
from trl import SFTConfig

lora_config = LoraConfig(
    r=HYPERPARAMS["lora_r"],
    lora_alpha=HYPERPARAMS["lora_alpha"],
    lora_dropout=HYPERPARAMS["lora_dropout"],
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["o_proj", "qkv_proj", "gate_up_proj", "down_proj"],
)

sft_config = SFTConfig(
    output_dir="./phi3-yoda-adapter",
    num_train_epochs=HYPERPARAMS["num_train_epochs"],
    learning_rate=HYPERPARAMS["learning_rate"],
    per_device_train_batch_size=HYPERPARAMS["per_device_train_batch_size"],
    max_seq_length=HYPERPARAMS["max_seq_length"],
    packing=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",  # "paged_adamw_8bit" in the cloud, needs bitsandbytes
    logging_steps=10,
    eval_strategy="epoch",
    report_to=["tensorboard"],
    logging_dir="./logs",
    seed=HYPERPARAMS["seed"],
)

print(lora_config)

/Users/andrianrajesy/Documents/Albert School/MSC2/ Agentic workflows/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type='CAUSAL_LM', inference_mode=False, r=8, target_modules={'o_proj', 'down_proj', 'gate_up_proj', 'qkv_proj'}, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False))


In [3]:
from accelerate import init_empty_weights
from peft import get_peft_model
from transformers import AutoConfig, AutoModelForCausalLM

model_config = AutoConfig.from_pretrained(HYPERPARAMS["model_id"])
with init_empty_weights():
    model = AutoModelForCausalLM.from_config(model_config)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 12,582,912 || all params: 3,833,662,464 || trainable%: 0.3282


In [4]:
print(peft_model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Phi3ForCausalLM(
      (model): Phi3Model(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
        (embed_dropout): Dropout(p=0.0, inplace=False)
        (layers): ModuleList(
          (0-31): 32 x Phi3DecoderLayer(
            (self_attn): Phi3SdpaAttention(
              (o_proj): lora.Linear(
                (base_layer): Linear(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_ve